<a href="https://colab.research.google.com/github/Miguithub/NPP_Cripto_5min/blob/Experimentos-112024-012026/r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# r — competencia de dos LSTM robustas
Único objetivo: **BTC__return_k1(t+h)** con información disponible al cierre de t.
Se eliminó el filtro por familias y toda experimentación sobre participación.
La explicación de dilución por grupos es una hipótesis plausible, no quedó demostrada por el resultado anterior.
Ahora se selecciona **top 15 por información mutua (MI)** dentro del entrenamiento y se depura
colinealidad por correlación y VIF, conservando prioridad por MI. No se rellena la lista después de depurar.

**Competencia:** tu LSTM compacta con Huber frente a una LSTM de dos escalas con LogCosh.
Mismas variables, etiquetas, fechas, ventana, preprocesamiento, semillas y presupuesto.
La alternativa cambia arquitectura y pérdida a la vez: se comparan recetas completas, no el efecto aislado de cada cambio.
No se garantiza que ninguna supere el baseline cero ni que la alternativa gane.

### Verificación de esta revisión
En la copia de BS.parquet revisada el 14/10/2026, el top 15 quedó en **7 variables** después de depurar:
`BTC__RLIQ`, `BTC_range_rel_proxy`, `BTC_v`, `BTC__rolling_volatility`, `YS`, `BTC__BILST`, `BTC__Arel`.
La notebook recalcula la selección al ejecutarse. MI puede detectar dependencia con la magnitud/volatilidad
sin aportar señal sobre el signo; por eso se mantiene el baseline cero.
Se probaron colinealidad por pares y múltiple, alineación temporal, entrenamiento corto de ambos modelos,
las pérdidas guardadas, reducción del LR y early stopping. El entrenamiento completo y la competencia
definitiva quedan para la ejecución de la notebook; no se presentan métricas de la prueba corta como resultados.


## 1. Entorno

In [1]:
%pip install -q "tensorflow>=2.16,<2.22" pyarrow pandas numpy scipy matplotlib scikit-learn statsmodels

In [2]:
try:
    from google.colab import drive
except ImportError:
    print('Entorno local: no se monta Google Drive.')
else:
    drive.mount('/content/drive')

Mounted at /content/drive


## 2. Todos los controles editables
Editar este panel **antes** de ejecutar el resto. Ejecutar todas las celdas para una nueva configuración.
`EVALUAR_TEST=False` evita mirar prueba durante la experimentación. Para abrirla después de congelar
decisiones, cambiar ese valor aquí, ejecutar esta celda y solo la última celda; las rutas de ejecución
se crean en la celda de lectura, no aquí.

`TOP_K_MI=15` incluye la historia del retorno si entra al ranking; no se fuerza ni agrega una 16.ª entrada.
El horizonte es futuro: MI contra r(t+h), nunca contra r(t). Los retardos de publicación son configurables.
Huber delta=1 se aplica al retorno normalizado por la desviación estándar de train; no significa 100% de retorno.

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, gc, warnings, hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from IPython.display import display
except ImportError:
    display = print


CARPETA_BS = Path('/content/drive/MyDrive/Neural/NPP/Cripto/Crypto 5 minutos/4. CD/BTC/BS')
ARCHIVO = CARPETA_BS / 'BS.parquet'
OBJETIVOS = ['BTC__return_k1']
HORIZONTE = 1
VENTANA = 72
FRACCION_TRAIN = 0.65
FRACCION_VAL = 0.20
FRECUENCIA = pd.Timedelta('5min')
LOTE_LECTURA = 8192
MAX_RAM_DF_MB = 512
RETRASOS = {}
EXCLUIR_MANUAL = []
MAX_FALTANTES = 0.10

# Selección: top K primero, depuración después. Ningún filtro por familias.
TOP_K_MI = 15
MAX_FILAS_MI = 20000
MI_REPETICIONES = 3
MI_VECINOS = 5
SEMILLA_SELECCION = 42
CORR_REDUNDANCIA = 0.90    # máximo de |Pearson| y |Spearman|, en train
VIF_MAX = 10.0            # también controla dependencias entre 3 o más variables
USAR_VIF = True

# Entrenamiento compartido para las dos recetas.
ENTRENAR = True
EVALUAR_TEST = False
SEMILLAS = [42, 123, 2026]
EPOCAS = 35
BATCH = 256
LEARNING_RATE = 1e-3
CLIPNORM = 1.0
PACIENCIA = 10
EARLY_MIN_DELTA = 0.0
MONITOR_MODE = 'min'
LR_FACTOR = 0.5
LR_PACIENCIA = 3
LR_MIN = 1e-5
LR_MIN_DELTA = 1e-4
HUBER_DELTA = 1.0

ARQUITECTURAS = {'BTC__return_k1': {
    'Usuario_Huber': dict(tipo='compacta', unidades=16, dropout=0.30, dense=8, l2=1e-3, loss='huber'),
    'Rival_multiescala': dict(tipo='doble', unidades=24, corta=12, unidades_corta=12,
                            dropout=0.25, dense=16, l2=1e-3, loss='logcosh'),
}}
# Para aislar arquitectura, se puede cambiar loss del rival a 'huber'.
CUANTIL_COLA = 0.95       # umbral de |r| estimado solo en train
BOOTSTRAP_BLOQUE = 288
BOOTSTRAP_REPETICIONES = 1000
GRAFICOS = True
MOSTRAR_FIGURAS = False
N_GRAFICOS = 12000
TEMPORALES = True
MAX_TRAMO_TESTS = 10000
LAGS_TESTS = 24
LAGS_ARCH = 12
LAGS_ACF = 48

## 3. Lectura por columnas y partición temporal
PyArrow lee por lotes. Se conservan fechas y se rechazan duplicados/huecos.
Se purgan los orígenes cuyas etiquetas cruzarían un borde. El test no interviene en selección ni escalado.
Las columnas del experimento anterior sobre participación no se cargan.

In [4]:
def cargar_df(path):
    with pq.ParquetFile(path, pre_buffer=False) as pf:
        schema = pf.schema_arrow
        numeric = [f.name for f in schema if pa.types.is_integer(f.type) or pa.types.is_floating(f.type)]
        columns = ['bar_end'] + [c for c in numeric if (c.startswith('BTC_') or c.startswith('YS') or c == 'SYS_imbalance') and not c.endswith('_yp')]
        missing = set(['bar_end', *OBJETIVOS]) - set(schema.names)
        if missing:
            raise ValueError(f'Faltan columnas: {missing}')
        estimate = pf.metadata.num_rows * len(columns) * 8 * 3 / 1e6
        if estimate > MAX_RAM_DF_MB:
            raise MemoryError(f'Estimación con copias {estimate:.1f} MB > {MAX_RAM_DF_MB} MB; reducir columnas.')
        # Preasignación: no se concatena una lista de dataframes ni ventanas 3D.
        arrays = {c: np.empty(pf.metadata.num_rows, dtype=np.float64) for c in columns if c != 'bar_end'}
        time = np.empty(pf.metadata.num_rows, dtype='datetime64[ns]')
        offset = 0
        for batch in pf.iter_batches(columns=columns, batch_size=LOTE_LECTURA, use_threads=False):
            end = offset + batch.num_rows
            for c in arrays:
                arrays[c][offset:end] = batch.column(c).to_numpy(zero_copy_only=False)
            time[offset:end] = batch.column('bar_end').to_numpy(zero_copy_only=False)
            offset = end
    data = pd.DataFrame(arrays, copy=False)
    data.insert(0, 'bar_end', pd.to_datetime(time, utc=True))
    return data

assert HORIZONTE >= 1 and VENTANA >= 2
assert 0 < FRACCION_TRAIN < 1 and 0 < FRACCION_VAL < 1 - FRACCION_TRAIN
df = cargar_df(ARCHIVO)
if df.bar_end.isna().any() or df.bar_end.duplicated().any() or not df.bar_end.is_monotonic_increasing:
    raise ValueError('Fechas inválidas, repetidas o desordenadas: revisar fuente, no ordenar silenciosamente.')
if not df.bar_end.diff().iloc[1:].eq(FRECUENCIA).all():
    raise ValueError('Hay huecos/cambios de frecuencia: regularizar explícitamente antes de crear ventanas.')
n = len(df)
b1, b2 = int(n * FRACCION_TRAIN), int(n * (FRACCION_TRAIN + FRACCION_VAL))
TRAIN_END = b1 - HORIZONTE
if TRAIN_END < 10 * VENTANA:
    raise ValueError('Entrenamiento demasiado corto para la ventana configurada.')
EJECUCION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SALIDA = CARPETA_BS / ('r_resultados_' + EJECUCION)
SALIDA.mkdir(parents=True, exist_ok=False)
Xraw = df.drop(columns='bar_end').replace([np.inf, -np.inf], np.nan)
for c, lag in RETRASOS.items():
    if c not in Xraw or not isinstance(lag, int) or lag < 0:
        raise ValueError(f'Retraso inválido: {c}={lag}')
    Xraw[c] = Xraw[c].shift(lag)
split_table = pd.DataFrame([
    ['train',0,TRAIN_END-1,b1-1], ['validacion',b1,b2-HORIZONTE-1,b2-1],
    ['prueba',b2,n-HORIZONTE-1,n-1]], columns=['particion','origen_desde','origen_hasta','ultima_etiqueta'])
display(split_table)
print('df:', df.shape, '| MB:', round(df.memory_usage(deep=True).sum()/1e6, 2))
print('Cobertura:', df.bar_end.iloc[0], '→', df.bar_end.iloc[-1])
split_table.to_csv(SALIDA/'particiones.csv', index=False)

UMBRAL_COLA = float(df[OBJETIVOS[0]].iloc[:b1].abs().quantile(CUANTIL_COLA))

,particion,origen_desde,origen_hasta,ultima_etiqueta
0,train,0,85548,85549
1,validacion,85550,111871,111872
2,prueba,111873,131614,131615


df: (131616, 44) | MB: 46.33
Cobertura: 2024-11-01 00:05:00+00:00 → 2026-02-01 00:00:00+00:00


## 4. Calidad y gráficos (solo entrenamiento)
Se excluyen constantes, faltantes excesivos, columnas de auditoría y posibles etiquetas futuras.
Se conservan los outliers: los cuatro paneles por variable muestran distribución, serie, Q-Q y diferencias.
Los diagnósticos ADF/KPSS/ARCH son exploratorios y se pueden desactivar con TEMPORALES.

In [5]:
def diagnosticar(data):
    rows = []
    for c in data:
        s = data[c].replace([np.inf,-np.inf],np.nan)
        v = s.dropna()
        q = v.quantile([.01,.25,.5,.75,.99])
        rows.append(dict(variable=c, faltantes=float(s.isna().mean()), infinitos=int(np.isinf(data[c]).sum()),
                         unicos=int(v.nunique()), media=v.mean(), std=v.std(), minimo=v.min(), maximo=v.max(),
                         p01=q.get(.01), mediana=q.get(.5), p99=q.get(.99),
                         asimetria=v.skew(), curtosis=v.kurt(), acf1=s.autocorr(1) if v.nunique()>1 else np.nan))
    return pd.DataFrame(rows).set_index('variable')

calidad = diagnosticar(df.drop(columns='bar_end').iloc[:TRAIN_END])
calidad.to_csv(SALIDA/'calidad_train.csv')
display(calidad)
# Estabilidad descriptiva entre tres tramos del entrenamiento, sin inspeccionar test.
estabilidad=[]
for bloque,ids in enumerate(np.array_split(np.arange(TRAIN_END),3),1):
    for c in Xraw:
        s=Xraw[c].iloc[ids]
        estabilidad.append(dict(bloque=bloque,variable=c,mediana=s.median(),
                                iqr=s.quantile(.75)-s.quantile(.25),faltantes=s.isna().mean()))
pd.DataFrame(estabilidad).to_csv(SALIDA/'estabilidad_train.csv',index=False)
excluidas = {}
for c in Xraw:
    s = Xraw[c].iloc[:TRAIN_END]
    if s.isna().mean() > MAX_FALTANTES:
        excluidas[c] = 'exceso de faltantes en train'
    elif s.nunique(dropna=True) <= 1:
        excluidas[c] = 'constante en train'
    elif c.endswith('_AUDIT') or re.search(r'target|lead|future|tplus', c, re.I):
        excluidas[c] = 'auditoría o posible etiqueta futura'
    elif c in EXCLUIR_MANUAL:
        excluidas[c] = 'exclusión manual'
CANDIDATAS = [c for c in Xraw if c not in excluidas]
for target in OBJETIVOS:
    if target not in CANDIDATAS:
        raise ValueError(f'Objetivo no utilizable: {target}; {excluidas.get(target)}')
display(pd.Series(excluidas, name='motivo'))
(SALIDA/'exclusiones.json').write_text(json.dumps(excluidas,indent=2),encoding='utf-8')

def guardar_fig(fig, name):
    fig.tight_layout()
    fig.savefig(SALIDA / (name+'.png'), dpi=120)
    if MOSTRAR_FIGURAS:
        display(fig)
    plt.close(fig)

if GRAFICOS:
    rng = np.random.default_rng(42)
    sample_rows = np.sort(rng.choice(TRAIN_END, min(TRAIN_END,N_GRAFICOS), replace=False))
    for c in Xraw:
        s = Xraw[c].iloc[:TRAIN_END]
        vals = s.iloc[sample_rows].dropna().to_numpy()
        delta = s.diff().iloc[sample_rows].dropna().to_numpy()
        fig, axes = plt.subplots(2,2,figsize=(11,6))
        if len(vals):
            axes[0,0].hist(vals,bins=60,density=True)
            axes[0,1].plot(df.bar_end.iloc[sample_rows],s.iloc[sample_rows],lw=.5)
            locator=mdates.AutoDateLocator(minticks=3,maxticks=5)
            axes[0,1].xaxis.set_major_locator(locator)
            axes[0,1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        if len(vals)>2 and np.ptp(vals)>0:
            stats.probplot(vals,plot=axes[1,0])
        if len(delta):
            axes[1,1].hist(delta,bins=60,density=True)
        for a,title in zip(axes.flat,['Distribución','Serie (muestra)','Q-Q normal','Diferencias consecutivas']):
            a.set_title(title)
        fig.suptitle(f'{c} | solo train | muestra {len(sample_rows):,}')
        guardar_fig(fig,'distribucion_'+c)

def tramo_continuo(s, max_rows=MAX_TRAMO_TESTS):
    # Mayor tramo sin nulos, sin juntar observaciones separadas por huecos.
    valid = s.notna()
    blocks = valid.ne(valid.shift()).cumsum()
    lengths = s[valid].groupby(blocks[valid]).size()
    if lengths.empty:
        return np.array([])
    return s[blocks.eq(lengths.idxmax())].iloc[-max_rows:].to_numpy()

def tests_temporales():
    from statsmodels.tsa.stattools import adfuller, kpss, acf
    from statsmodels.stats.diagnostic import het_arch
    report = []
    for c in CANDIDATAS:
        a = tramo_continuo(Xraw[c].iloc[:TRAIN_END])
        row = {'variable':c, 'n_tramo':len(a)}
        if len(a) >= 200 and np.std(a)>0:
            z = (a-a.mean())/a.std()
            for name, func in {
                'adf_p':lambda:adfuller(z,maxlag=LAGS_TESTS,autolag='AIC')[1],
                'kpss_p':lambda:kpss(z,regression='c',nlags='auto')[1],
                'arch_p':lambda:het_arch(np.diff(z),nlags=LAGS_ARCH)[1],
            }.items():
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        row[name] = func()
                except (ValueError,np.linalg.LinAlgError) as e:
                    row[name+'_error'] = str(e)
            if c in OBJETIVOS and GRAFICOS:
                from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
                fig, ax = plt.subplots(1,2,figsize=(11,3))
                plot_acf(z,lags=LAGS_ACF,ax=ax[0]); plot_pacf(z,lags=LAGS_ACF,ax=ax[1],method='ywm')
                fig.suptitle(c+' | tramo continuo train')
                guardar_fig(fig,'acf_pacf_'+c)
        report.append(row)
    return pd.DataFrame(report)

temporal = tests_temporales() if TEMPORALES else pd.DataFrame()
temporal.to_csv(SALIDA/'diagnosticos_temporales_train.csv',index=False)
display(temporal)

,faltantes,infinitos,unicos,media,std,minimo,maximo,p01,mediana,p99,asimetria,curtosis,acf1
variable,,,,,,,,,,,,,
YS_growth,0.000012,0,85493,4.187970e-06,3.816383e-01,-1.886099e+00,2.923294e+00,-8.852089e-01,-2.070910e-02,1.069467e+00,0.493161,2.020949,-0.324223
YS,0.000000,0,85433,4.960611e+07,5.203695e+07,3.424687e+06,1.376932e+09,8.123651e+06,3.490539e+07,2.454868e+08,5.874771,70.149726,0.816334
YS_volatility,0.000842,0,84899,3.778314e-01,5.722940e-02,1.928558e-01,5.512010e-01,2.386877e-01,3.848300e-01,4.867131e-01,-0.494809,-0.158661,0.999518
SYS_imbalance,0.000000,0,85508,-1.066979e-02,1.687667e-01,-8.402497e-01,8.013449e-01,-4.123731e-01,-9.458195e-03,3.990877e-01,0.015213,0.490832,0.190452
BTC_y,0.000000,0,85447,8.521884e+06,1.217051e+07,9.678484e+04,5.149494e+08,6.695547e+05,5.006172e+06,5.630345e+07,7.823602,137.120286,0.718776
BTC_v,0.000000,0,85269,8.875342e+01,1.300680e+02,9.095800e-01,5.471439e+03,6.545457e+00,5.081882e+01,5.988722e+02,7.646764,129.176242,0.730304
BTC_observed,0.000000,0,1,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000,0.000000,NaN
BTC_active_trading,0.000000,0,1,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000,0.000000,NaN
BTC_turnover_usdt_proxy,0.000000,0,85451,8.521475e+06,1.216699e+07,9.678485e+04,5.189033e+08,6.694882e+05,5.006122e+06,5.630810e+07,7.824930,137.670322,0.719049


,motivo
BTC_observed,constante en train
BTC_active_trading,constante en train
BTC__CA,constante en train
BTC__CO,constante en train
BTC__FLOW_TOTAL_AUDIT,auditoría o posible etiqueta futura
BTC__TURNOVER_PER_TRADE_AUDIT,auditoría o posible etiqueta futura


,variable,n_tramo,adf_p,kpss_p,arch_p
0,YS_growth,10000,0.000000e+00,0.100000,0.000000e+00
1,YS,10000,3.451405e-21,0.059965,1.187933e-106
2,YS_volatility,10000,4.187590e-02,0.010000,2.958320e-03
3,SYS_imbalance,10000,0.000000e+00,0.100000,5.890089e-177
4,BTC_y,10000,8.745532e-25,0.100000,0.000000e+00
5,BTC_v,10000,7.042002e-25,0.100000,0.000000e+00
6,BTC_turnover_usdt_proxy,10000,8.980291e-24,0.100000,0.000000e+00
7,BTC_range_rel_proxy,10000,9.944937e-21,0.100000,0.000000e+00
8,BTC_illiquidity_proxy,10000,2.483767e-24,0.100000,0.000000e+00
9,BTC_activity_trades_proxy,10000,2.149306e-21,0.013041,0.000000e+00


## 5. Preprocesamiento y métricas
Mediana, IQR y asinh ajustados con train; no se recortan retornos extremos ni se imputan etiquetas.
Para comparar pérdidas se normaliza y con media/desviación de train, igual en ambos modelos.
Además de MAE/RMSE/R², se informa MAE en |r| mayor al percentil de train, cantidad de casos y acierto de signo.
MAE_COLA no participa en la selección de features ni modifica las etiquetas.

In [6]:
class Preparador:
    def fit(self, frame):
        a = np.asarray(frame,dtype=float)
        self.median = np.nanmedian(a,axis=0)
        if not np.isfinite(self.median).all():
            raise ValueError('Alguna variable no tiene valores finitos en este entrenamiento.')
        q = np.nanpercentile(a,[25,75],axis=0)
        self.scale = q[1]-q[0]
        fallback = np.nanstd(a,axis=0)
        self.scale = np.where(self.scale>0,self.scale,np.where(fallback>0,fallback,1.0))
        return self
    def transform(self, frame):
        a = np.asarray(frame,dtype=float)
        a = np.where(np.isfinite(a),a,self.median)
        result = np.arcsinh((a-self.median)/self.scale)
        if not np.isfinite(result).all():
            raise ValueError('Transformación produjo valores no finitos.')
        return result.astype('float32')

def subindices(idx, limit):
    return idx[np.linspace(0,len(idx)-1,min(limit,len(idx)),dtype=int)]


def metricas(y,p):
    y,p=np.asarray(y),np.asarray(p)
    tail=np.abs(y)>=UMBRAL_COLA
    return dict(MAE=float(mean_absolute_error(y,p)), RMSE=float(np.sqrt(mean_squared_error(y,p))),
                R2=float(r2_score(y,p)), acierto_signo=float((np.sign(y)==np.sign(p)).mean()),
                MAE_COLA=float(np.mean(np.abs(y[tail]-p[tail]))) if tail.any() else np.nan,
                n_cola=int(tail.sum()))

## 6. Top 15 por MI → correlación → VIF
Se promedia MI sobre muestras reproducibles de train; su dispersión queda en el reporte.
La MI detecta dependencia, no garantiza predicción fuera de muestra. No hay Ridge/ExtraTrees ni filtro por grupo.
Se recorre el top 15 de mayor a menor MI: si una candidata se solapa por correlación con una ya aceptada,
se descarta la candidata (menor MI). También se rechaza si al agregarla aparece VIF excesivo en el conjunto.
Este criterio conserva prioridad por MI incluso en dependencias de tres o más columnas.
El VIF se calcula sobre columnas originales estandarizadas e imputadas con la mediana de train.
Se reportan ranking completo, top 15, pares correlacionados, decisiones y VIF final.

In [7]:
def calcular_vif(frame):
    a=np.asarray(frame,dtype=float)
    a=np.where(np.isfinite(a),a,np.nanmedian(a,axis=0))
    std=a.std(axis=0)
    if np.any(std==0):
        raise ValueError('VIF: columna constante.')
    z=(a-a.mean(axis=0))/std
    result={}
    for j,c in enumerate(frame.columns):
        others=np.delete(z,j,axis=1)
        residual=z[:,j] if others.shape[1]==0 else z[:,j]-others@np.linalg.lstsq(others,z[:,j],rcond=None)[0]
        ratio=float(np.mean(residual**2))
        result[c]=np.inf if ratio<1e-12 else 1/ratio
    return pd.Series(result,name='VIF')

def depurar_top(top, frame):
    names=list(top.index)
    corr_p=frame[names].corr(method='pearson').abs()
    corr_s=frame[names].corr(method='spearman').abs()
    correlation=corr_p.combine(corr_s,np.maximum)
    accepted=[]; decisions=[]
    for c in names:
        overlap=[s for s in accepted if correlation.loc[c,s]>=CORR_REDUNDANCIA]
        if overlap:
            decisions.append(dict(variable=c,aceptada=False,motivo='correlacion',con=','.join(overlap)))
            continue
        vf=calcular_vif(frame[accepted+[c]]) if USAR_VIF else pd.Series(dtype=float)
        if len(vf) and vf.max()>VIF_MAX:
            decisions.append(dict(variable=c,aceptada=False,motivo='VIF conjunto',con=','.join(vf[vf>VIF_MAX].index)))
            continue
        accepted.append(c)
        decisions.append(dict(variable=c,aceptada=True,motivo='conservada por prioridad MI',con=''))
    return accepted,pd.DataFrame(decisions),correlation

target=OBJETIVOS[0]
assert TOP_K_MI>=1 and MI_REPETICIONES>=1 and MI_VECINOS>=1
assert 0<CORR_REDUNDANCIA<=1 and VIF_MAX>1
labels=df[target].shift(-HORIZONTE).iloc[:TRAIN_END]
frame=Xraw[CANDIDATAS].iloc[:TRAIN_END]
valid=np.flatnonzero(np.isfinite(labels.to_numpy()))
if len(valid)<=MI_VECINOS+1:
    raise ValueError('No hay suficientes etiquetas para MI.')
mi_runs=[]
for rep in range(MI_REPETICIONES):
    rng=np.random.default_rng(SEMILLA_SELECCION+rep)
    idx=np.sort(rng.choice(valid,size=min(MAX_FILAS_MI,len(valid)),replace=False))
    sample=frame.iloc[idx]
    usable=[c for c in frame if sample[c].notna().any()]
    values=Preparador().fit(sample[usable]).transform(sample[usable])
    mi=mutual_info_regression(values,labels.iloc[idx].to_numpy(),n_neighbors=MI_VECINOS,
                             random_state=SEMILLA_SELECCION+rep)
    mi_runs.append(pd.Series(mi,index=usable).reindex(frame.columns))
mi_table=pd.DataFrame(mi_runs)
ranking=pd.DataFrame({'MI':mi_table.mean(),'MI_std':mi_table.std(ddof=0)}).dropna(subset=['MI'])
ranking=ranking.sort_values('MI',ascending=False,kind='stable')
top=ranking.head(TOP_K_MI)
selected,decisions,correlation=depurar_top(top,frame)
if not selected:
    raise ValueError('No quedaron variables; revisar calidad y umbrales.')
SELECCION={target:selected}
ranking.to_csv(SALIDA/'ranking_MI_train.csv')
top.to_csv(SALIDA/'top15_MI_antes_colinealidad.csv')
decisions.to_csv(SALIDA/'decisiones_colinealidad.csv',index=False)
correlation.to_csv(SALIDA/'correlacion_top15.csv')
vif_final=calcular_vif(frame[selected])
vif_final.to_csv(SALIDA/'VIF_final.csv')
(SALIDA/'features.json').write_text(json.dumps(SELECCION,indent=2),encoding='utf-8')
display(top); display(decisions); display(vif_final)
print('Mismas entradas para ambos modelos:',selected)
if GRAFICOS:
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    top.MI.sort_values().plot.barh(ax=axes[0],title='Top 15 MI contra retorno futuro (train)')
    im=axes[1].imshow(correlation,vmin=0,vmax=1,cmap='viridis')
    axes[1].set_xticks(range(len(top)),top.index,rotation=90)
    axes[1].set_yticks(range(len(top)),top.index)
    fig.colorbar(im,ax=axes[1]); guardar_fig(fig,'seleccion_MI_colinealidad')

,MI,MI_std
BTC__RLIQ,0.127779,0.004067
BTC_activity_trades_proxy,0.126413,0.005092
BTC_range_rel_proxy,0.125042,0.002184
BTC__ABS,0.114177,0.001018
BTC__ABS_S,0.113667,0.002539
BTC_v,0.105990,0.001983
BTC_y,0.099878,0.003359
BTC_turnover_usdt_proxy,0.099721,0.003273
BTC_taker_buy_usdt_proxy,0.094442,0.003675
BTC_taker_sell_usdt_proxy,0.093300,0.002948


,variable,aceptada,motivo,con
0,BTC__RLIQ,True,conservada por prioridad MI,
1,BTC_activity_trades_proxy,False,correlacion,BTC__RLIQ
2,BTC_range_rel_proxy,True,conservada por prioridad MI,
3,BTC__ABS,False,correlacion,BTC_range_rel_proxy
4,BTC__ABS_S,False,correlacion,BTC_range_rel_proxy
5,BTC_v,True,conservada por prioridad MI,
6,BTC_y,False,correlacion,BTC_v
7,BTC_turnover_usdt_proxy,False,correlacion,BTC_v
8,BTC_taker_buy_usdt_proxy,False,correlacion,BTC_v
9,BTC_taker_sell_usdt_proxy,False,correlacion,BTC_v


,VIF
BTC__RLIQ,5.223178
BTC_range_rel_proxy,3.925515
BTC_v,4.634085
BTC__rolling_volatility,2.327853
YS,4.736857
BTC__BILST,2.794883
BTC__Arel,2.866629


Mismas entradas para ambos modelos: ['BTC__RLIQ', 'BTC_range_rel_proxy', 'BTC_v', 'BTC__rolling_volatility', 'YS', 'BTC__BILST', 'BTC__Arel']


## 7. Los dos competidores
**Usuario_Huber:** se conserva la LSTM compacta regularizada (16 unidades), ahora con Huber(delta=1).
**Rival_multiescala:** dos ramas LSTM: una ve las 72 barras y otra solo las últimas 12; sus estados se
normalizan y combinan. LogCosh se comporta de forma cuadrática cerca de cero y aproximadamente lineal
en errores grandes; su gradiente es acotado. La arquitectura intenta separar memoria lenta de movimientos recientes.
Es una hipótesis de diseño, no una ventaja demostrada. Mi preferencia dependerá del MAE y del error en colas
de validación; la red compacta sigue siendo la referencia de menor complejidad.

Ambas reciben ReduceLROnPlateau(val_mae, factor=.5, patience=3, min_lr=1e-5) y EarlyStopping con paciencia 10.
Huber y LogCosh reducen la influencia de outliers sobre el ajuste, pero pueden subestimar extremos; por eso se reporta MAE_COLA.

In [8]:
def construir_modelo(shape,cfg):
    import tensorflow as tf
    k=tf.keras
    regularizer=k.regularizers.l2(cfg['l2'])
    inputs=k.Input(shape=shape)
    if cfg['tipo']=='compacta':
        x=k.layers.LSTM(cfg['unidades'],kernel_regularizer=regularizer)(inputs)
    elif cfg['tipo']=='doble':
        short=cfg['corta']
        if not 1<=short<=shape[0]:
            raise ValueError('La ventana corta debe estar dentro de VENTANA.')
        recent=k.layers.Cropping1D(cropping=(shape[0]-short,0))(inputs)
        long_state=k.layers.LSTM(cfg['unidades'],kernel_regularizer=regularizer)(inputs)
        short_state=k.layers.LSTM(cfg['unidades_corta'],kernel_regularizer=regularizer)(recent)
        x=k.layers.Concatenate()([k.layers.LayerNormalization()(long_state),
                                  k.layers.LayerNormalization()(short_state)])
    else:
        raise ValueError('Arquitectura desconocida.')
    x=k.layers.Dropout(cfg['dropout'])(x)
    x=k.layers.Dense(cfg['dense'],activation='relu',kernel_regularizer=regularizer)(x)
    out=k.layers.Dense(1)(x)
    model=k.Model(inputs,out)
    losses={'huber':k.losses.Huber(delta=HUBER_DELTA),'logcosh':k.losses.LogCosh()}
    model.compile(optimizer=k.optimizers.Adam(LEARNING_RATE,clipnorm=CLIPNORM),
                  loss=losses[cfg['loss']],metrics=['mae'])
    return model

def callbacks_entrenamiento():
    import tensorflow as tf
    k=tf.keras
    class RegistrarLR(k.callbacks.Callback):
        def on_epoch_end(self,epoch,logs=None):
            if logs is not None:
                logs['learning_rate']=float(k.backend.get_value(self.model.optimizer.learning_rate))
    return [k.callbacks.ReduceLROnPlateau(monitor='val_mae',mode=MONITOR_MODE,factor=LR_FACTOR,patience=LR_PACIENCIA,
                  min_lr=LR_MIN,min_delta=LR_MIN_DELTA,verbose=1),
            k.callbacks.EarlyStopping(monitor='val_mae',mode=MONITOR_MODE,patience=PACIENCIA,
                  min_delta=EARLY_MIN_DELTA,restore_best_weights=True),
            RegistrarLR(),k.callbacks.TerminateOnNaN()]

## 8. Ventanas en lotes
PyDataset materializa un lote de ventanas; no se construye el tensor completo en RAM. Las dos redes comparten las mismas ventanas.

In [9]:
def preparar_objetivo(target):
    features=SELECCION[target]
    prep=Preparador().fit(Xraw[features].iloc[:TRAIN_END])
    x=prep.transform(Xraw[features])
    y=df[target].shift(-HORIZONTE).to_numpy(dtype=float)
    good=np.isfinite(y)
    origins=np.arange(n)
    sets={
        'train':origins[(origins>=VENTANA-1)&(origins<TRAIN_END)&good],
        'val':origins[(origins>=max(b1,VENTANA-1))&(origins<b2-HORIZONTE)&good],
        'test':origins[(origins>=max(b2,VENTANA-1))&(origins<n-HORIZONTE)&good],
    }
    if any(len(v)==0 for v in sets.values()):
        raise ValueError('Partición sin ventanas válidas.')
    mu,sd=y[sets['train']].mean(),y[sets['train']].std()
    if not sd>0:
        raise ValueError('Objetivo constante.')
    np.savez(SALIDA/f'{target}_preprocesamiento.npz',mediana=prep.median,escala=prep.scale,y_mu=mu,y_sd=sd)
    return x,y,sets,mu,sd

def dataset(x,y,origins,mu,sd):
    import tensorflow as tf
    class Ventanas(tf.keras.utils.PyDataset):
        def __init__(self):
            super().__init__(workers=1,use_multiprocessing=False,max_queue_size=1)
        def __len__(self):
            return int(np.ceil(len(origins)/BATCH))
        def __getitem__(self,batch_index):
            if batch_index<0 or batch_index>=len(self):
                raise IndexError(batch_index)
            start=batch_index*BATCH
            idx=origins[start:start+BATCH]
            xb=np.stack([x[t-VENTANA+1:t+1] for t in idx]).astype('float32')
            yb=((y[idx]-mu)/sd).astype('float32')[:,None]
            return xb,yb
    return Ventanas()

def predicciones(model,x,y,origins,mu,sd):
    # Un lote por llamada; evita copias gigantes y conserva el orden de orígenes.
    pieces=[]
    batches=dataset(x,y,origins,mu,sd)
    for i in range(len(batches)):
        xb,_=batches[i]
        pieces.append(model(xb,training=False).numpy().ravel())
    return np.concatenate(pieces)*sd+mu

## 9. Entrenamiento y competencia en validación
Se comparan MAE medio y dispersión entre las mismas semillas; no se elige la mejor semilla.
El ganador se fija por MAE medio de validación. Revisar también MAE_COLA antes de concluir superioridad.
Los baselines son cero, media histórica y persistencia. Si ninguna red mejora, se informa expresamente.
La predicción final de cada receta es el promedio de sus semillas. Las curvas incluyen val_mae y un CSV con learning_rate.

In [ ]:
RESULTADOS=[]
MODELOS_GUARDADOS={}
GANADORES={}
PRED_VAL={}
if ENTRENAR:
    import tensorflow as tf
    tf.config.experimental.enable_op_determinism()
    for target in OBJETIVOS:
        x,y,sets,mu,sd=preparar_objetivo(target)
        ids=sets['val']
        actual=y[ids]
        baseline={
            'media_train':np.full(len(ids),mu),
            'persistencia':df[target].to_numpy()[ids],
        }
        if target=='BTC__return_k1':
            baseline['cero']=np.zeros(len(ids))
        for name,p in baseline.items():
            valid=np.isfinite(p)
            RESULTADOS.append(dict(objetivo=target,modelo=name,semilla=-1,**metricas(actual[valid],p[valid])))
        PRED_VAL[target]={}
        for name,cfg in ARQUITECTURAS[target].items():
            paths=[]; preds=[]
            for seed in SEMILLAS:
                tf.keras.backend.clear_session()
                tf.keras.utils.set_random_seed(seed)
                model=construir_modelo((VENTANA,x.shape[1]),cfg)
                model.summary()
                history=model.fit(dataset(x,y,sets['train'],mu,sd),
                    validation_data=dataset(x,y,ids,mu,sd),epochs=EPOCAS,verbose=2,shuffle=False,
                    callbacks=callbacks_entrenamiento())
                if not all(np.isfinite(history.history['loss'])):
                    raise RuntimeError('Entrenamiento divergente; revisar datos y learning rate.')
                p=predicciones(model,x,y,ids,mu,sd)
                if not np.isfinite(p).all():
                    raise RuntimeError('Predicciones no finitas.')
                best_epoch=int(np.argmin(history.history['val_mae']))+1
                RESULTADOS.append(dict(objetivo=target,modelo=name,semilla=seed,epoca=best_epoch,
                                       parametros=model.count_params(),**metricas(actual,p)))
                path=SALIDA/f'{name}_seed{seed}.keras'
                model.save(path); paths.append(str(path)); preds.append(p)
                pd.DataFrame(history.history).to_csv(SALIDA/f'{name}_seed{seed}_historia.csv',index=False)
                if GRAFICOS:
                    fig,ax=plt.subplots(figsize=(7,3))
                    pd.DataFrame(history.history)[['mae','val_mae']].plot(ax=ax,title=f'{name} seed {seed} | escala normalizada')
                    guardar_fig(fig,f'{name}_seed{seed}_aprendizaje')
                del model; gc.collect()
            MODELOS_GUARDADOS[name]=paths
            PRED_VAL[target][name]=np.mean(preds,axis=0)
        result=pd.DataFrame(RESULTADOS)
        summary=result[(result.objetivo==target)&(result.semilla>=0)].groupby('modelo').agg(
            MAE_media=('MAE','mean'),MAE_std=('MAE','std'),RMSE_media=('RMSE','mean'),MAE_cola_media=('MAE_COLA','mean'))
        winner=summary.MAE_media.idxmin(); GANADORES[target]=winner
        display(summary)
        best_base=result[(result.objetivo==target)&(result.semilla<0)].sort_values('MAE').iloc[0]
        improvement=1-summary.loc[winner,'MAE_media']/best_base.MAE
        print(f'Recomendación por validación: {winner}. Mejora vs mejor baseline {best_base.modelo}: {improvement:.2%}.')
        if improvement<=0:
            print('Ninguna ventaja sobre baseline: preferir baseline; no afirmar capacidad predictiva superior.')
        pd.DataFrame({'fecha_objetivo':df.bar_end.iloc[ids+HORIZONTE].to_numpy(),
                      'real':actual,**PRED_VAL[target]}).to_csv(SALIDA/f'{target}_validacion_predicciones.csv',index=False)
        if GRAFICOS:
            fig,ax=plt.subplots(figsize=(12,3))
            for name,p in PRED_VAL[target].items():
                ax.plot(df.bar_end.iloc[ids[-1000:]+HORIZONTE],p[-1000:],label=name,alpha=.7)
            ax.plot(df.bar_end.iloc[ids[-1000:]+HORIZONTE],actual[-1000:],label='real',color='black',lw=.8)
            ax.legend(); guardar_fig(fig,target+'_validacion')
        del x; gc.collect()
    pd.DataFrame(RESULTADOS).to_csv(SALIDA/'metricas_validacion.csv',index=False)
else:
    print('Entrenamiento desactivado. Se completaron los diagnósticos y la selección.')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,681 (6.57 KB)

 Trainable params: 1,681 (6.57 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 72, 7)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cropping1d          │ (None, 12, 7)     │          0 │ input_layer[0][0] │
│ (Cropping1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 24)        │      3,072 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 12)        │        960 │ cropping1d[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 24)        │         48 │ lstm[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 12)        │         24 │ lstm_1[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 36)        │          0 │ layer_normalizat… │
│ (Concatenate)       │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 36)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        592 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         17 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,713 (18.41 KB)

 Trainable params: 4,713 (18.41 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/35
334/334 - 24s - 72ms/step - loss: 0.3775 - mae: 0.7048 - val_loss: 0.2870 - val_mae: 0.5811 - learning_rate: 0.0010
Epoch 2/35
334/334 - 22s - 66ms/step - loss: 0.3287 - mae: 0.6494 - val_loss: 0.2752 - val_mae: 0.5798 - learning_rate: 0.0010
Epoch 3/35
334/334 - 18s - 55ms/step - loss: 0.3182 - mae: 0.6466 - val_loss: 0.2681 - val_mae: 0.5793 - learning_rate: 0.0010
Epoch 4/35
334/334 - 22s - 67ms/step - loss: 0.3123 - mae: 0.6459 - val_loss: 0.2638 - val_mae: 0.5794 - learning_rate: 0.0010
Epoch 5/35
334/334 - 19s - 57ms/step - loss: 0.3086 - mae: 0.6457 - val_loss: 0.2608 - val_mae: 0.5793 - learning_rate: 0.0010
Epoch 6/35

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
334/334 - 18s - 55ms/step - loss: 0.3061 - mae: 0.6457 - val_loss: 0.2586 - val_mae: 0.5793 - learning_rate: 5.0000e-04
Epoch 7/35
334/334 - 19s - 56ms/step - loss: 0.3045 - mae: 0.6456 - val_loss: 0.2577 - val_mae: 0.5792 - learning_rate: 5.0000e-04
Epoch 8/35
334/334 - 22s -

## 10. Prueba final y reproducibilidad
EVALUAR_TEST se controla en el panel central. Tras congelar decisiones, se evalúa solo el ganador
de validación con su promedio de semillas; no se selecciona arquitectura con test.
Se guarda el umbral de colas y la configuración junto con modelos, predicciones y métricas.

In [ ]:
manifest=dict(archivo=str(ARCHIVO),filas=n,features=SELECCION,objetivo=OBJETIVOS,
    horizonte=HORIZONTE,ventana=VENTANA,b1=b1,b2=b2,semillas=SEMILLAS,
    arquitecturas=ARQUITECTURAS,ganadores=GANADORES,modelos=MODELOS_GUARDADOS,
    umbral_cola=UMBRAL_COLA,cuantil_cola=CUANTIL_COLA,retrasos=RETRASOS,
    top_k_mi=TOP_K_MI,mi_vecinos=MI_VECINOS,mi_repeticiones=MI_REPETICIONES,max_filas_mi=MAX_FILAS_MI,
    correlacion=CORR_REDUNDANCIA,usar_vif=USAR_VIF,vif_max=VIF_MAX,
    learning_rate=LEARNING_RATE,lr_factor=LR_FACTOR,lr_patience=LR_PACIENCIA,lr_min=LR_MIN,
    early_patience=PACIENCIA,epocas=EPOCAS,batch=BATCH,huber_delta=HUBER_DELTA,clipnorm=CLIPNORM,
    numpy=np.__version__,pandas=pd.__version__,pyarrow=pa.__version__)
(SALIDA/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
def intervalo_mejora(y,p,base,block=BOOTSTRAP_BLOQUE,reps=BOOTSTRAP_REPETICIONES):
    difference=np.abs(y-base)-np.abs(y-p)
    blocks=[difference[i:i+block] for i in range(0,len(y)-block+1,block)]
    if len(blocks)<5:
        return [np.nan,np.nan]
    means=np.array([v.mean() for v in blocks])
    rng=np.random.default_rng(42)
    samples=means[rng.integers(0,len(means),size=(reps,len(means)))].mean(axis=1)
    return np.quantile(samples,[.025,.975]).tolist()

if EVALUAR_TEST:
    if not GANADORES:
        raise ValueError('Primero entrenar y fijar ganadores en validación.')
    final_rows=[]
    for target,winner in GANADORES.items():
        x,y,sets,mu,sd=preparar_objetivo(target)
        ids=sets['test']; preds=[]
        for path in MODELOS_GUARDADOS[winner]:
            tf.keras.backend.clear_session()
            model=tf.keras.models.load_model(path)
            preds.append(predicciones(model,x,y,ids,mu,sd))
            del model
        p=np.mean(preds,axis=0); actual=y[ids]
        base_options={'media_train':np.full(len(ids),mu),'persistencia':df[target].to_numpy()[ids]}
        if target=='BTC__return_k1':
            base_options['cero']=np.zeros(len(ids))
        validation=pd.DataFrame(RESULTADOS)
        base_name=validation[(validation.objetivo==target)&(validation.semilla<0)].sort_values('MAE').iloc[0].modelo
        for name,bp in {winner:p,**base_options}.items():
            valid=np.isfinite(bp)
            row=dict(objetivo=target,modelo=name,**metricas(actual[valid],bp[valid]))
            if target=='BTC__return_k1':
                row['acierto_signo']=float((np.sign(actual[valid])==np.sign(bp[valid])).mean())
            final_rows.append(row)
        base=base_options[base_name]; valid=np.isfinite(base)
        print(target,winner,'IC95% mejora MAE vs',base_name,intervalo_mejora(actual[valid],p[valid],base[valid]))
        pd.DataFrame({'fecha_objetivo':df.bar_end.iloc[ids+HORIZONTE].to_numpy(),'real':actual,
                      'prediccion':p,'baseline':base}).to_csv(SALIDA/f'{target}_test_predicciones.csv',index=False)
        del x; gc.collect()
    display(pd.DataFrame(final_rows))
    pd.DataFrame(final_rows).to_csv(SALIDA/'metricas_test.csv',index=False)
else:
    print('Prueba sellada: no se calcularon métricas de test.')
print('Resultados guardados en:',SALIDA)

## Referencias
- [Huber y LogCosh — Keras](https://keras.io/api/losses/regression_losses/)
- [ReduceLROnPlateau — Keras](https://keras.io/api/callbacks/reduce_lr_on_plateau/)
La MI es un criterio exploratorio; conservar una variable no demuestra causalidad ni rentabilidad.